In [21]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_clientes = 60000

# 4 anos = 4 probabilidades (75% acumulados de 2022 a 2024 para treino, 25% em 2025 para teste)
safra_default = np.random.choice(
    [2022, 2023, 2024, 2025],
    size=n_clientes,
    p=[0.20, 0.25, 0.30, 0.25]
)

# Dados cadastrais e do cartão observados 12 MESES ANTES do default
renda_mensal = np.random.lognormal(mean=8.2, sigma=0.5, size=n_clientes).round(2)
limite_12m = (renda_mensal * np.random.uniform(1.5, 4.0, size=n_clientes)).round(2)

# Taxa de utilização 12m antes (alguns clientes já estão com 100% do limite tomado)
utilizacao_inicial = np.clip(np.random.beta(a=2, b=3, size=n_clientes) * 1.15, 0.0, 1.0)
saldo_12m = (limite_12m * utilizacao_inicial).round(2)

# Sinais comportamentais observados 12m antes
variacao_saldo_3m = np.random.normal(loc=0.15, scale=0.25, size=n_clientes).round(4)
qtd_atrasos_curtos_6m = np.random.poisson(lam=0.8, size=n_clientes)
pagou_minimo_ult_fatura = np.random.binomial(n=1, p=0.35, size=n_clientes)

# Simulando o default em 12 meses (0 = pagou em dia, 1 = calote)
logit_pd = (
    -3.5
    + 0.65 * qtd_atrasos_curtos_6m
    + 1.8 * utilizacao_inicial
    + 1.6 * np.clip(variacao_saldo_3m, -0.5, 1.0)
    + 0.45 * pagou_minimo_ult_fatura
    + np.random.normal(0, 0.5, size=n_clientes)
)
prob_default = 1 / (1 + np.exp(-logit_pd))
default_12m = np.random.binomial(n=1, p=prob_default)

# Simulando o comportamento real até o calote (ruído calibrado: 0.12 -> 0.07)
score_estresse = (
    0.30 * utilizacao_inicial
    + 0.25 * np.clip(variacao_saldo_3m, -0.5, 1.0)
    + 0.12 * qtd_atrasos_curtos_6m
    + 0.20 * pagou_minimo_ult_fatura
    + np.random.normal(0, 0.07, size=n_clientes)
)
ccf_verdadeiro = np.clip(score_estresse, 0.0, 1.0)

# Saldo real registrado no dia em que o cliente deu o default (0.0 se não deu default)
limite_livre_12m = limite_12m - saldo_12m
saldo_default = (
    saldo_12m + ccf_verdadeiro * limite_livre_12m + np.random.normal(0, 25, size=n_clientes)
).round(2)
saldo_default = np.maximum(saldo_default, 0.0)
saldo_default = np.where(default_12m == 1, saldo_default, 0.0)

df = pd.DataFrame({
    "safra_obs": safra_default,
    "renda_mensal": renda_mensal,
    "limite": limite_12m,
    "saldo": saldo_12m,
    "variacao_saldo_3m": variacao_saldo_3m,
    "qtd_atrasos_curtos_6m": qtd_atrasos_curtos_6m,
    "pagou_minimo_ult_fatura": pagou_minimo_ult_fatura,
    "saldo_default": saldo_default,
    "default_12m": default_12m,
})

df.to_csv("Dados.csv", index=False)